# LangChain Optimization Strategies Notebook

This notebook shows concrete cost and latency optimization techniques for agentic systems built with LangChain.

Topics covered:
- Prompt caching (prefix caching)
- Semantic caching with vector search
- Prompt compression (LLMLingua)
- Concise instruction patterns
- Output format and output-length constraints
- Task-based model routing
- Batch processing

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import yfinance as yf
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings

try:
    from llmlingua import PromptCompressor
except Exception:
    PromptCompressor = None

## 0) Configuration

Adjust model names and paths before running.

`RAG_VECTOR_DB_DIR` reuses the vector store directory from the RAG tutorial.

In [ ]:
MODEL_SMALL = 'llama3.1:8b'
MODEL_LARGE = 'llama3.1:70b'
EMBED_MODEL = os.getenv('OLLAMA_EMBEDDING_MODEL', 'nomic-embed-text')
OLLAMA_BASE_URL = os.getenv('OLLAMA_ENDPOINT', 'http://localhost:11434/v1')

PROJECT_ROOT = Path.cwd().parents[2] if 'langchain_financial_agent' in str(Path.cwd()) else Path.cwd()
RAG_VECTOR_DB_DIR = PROJECT_ROOT / 'ai_tutorials' / 'rag' / 'vector_db'
LOCAL_CACHE_DIR = Path.cwd() / 'optimization_cache'
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('RAG vector DB directory:', RAG_VECTOR_DB_DIR)
print('Local optimization cache:', LOCAL_CACHE_DIR)

## 1) Prompt Caching (Prefix Caching)

Put static content first so prompt-prefix caching can reuse computation:

- system role
- fixed API/tool instructions
- static schemas
- then dynamic user-specific fields

In [ ]:
STATIC_PREFIX = '''
You are a financial agent.
Follow output schema exactly:
{
  "ticker": string,
  "summary": string,
  "risks": string[]
}
Always ground responses in provided evidence only.
'''

def prefix_cache_key(static_prefix: str) -> str:
    return hashlib.sha256(static_prefix.encode('utf-8')).hexdigest()

def build_prompt(static_prefix: str, ticker: str, question: str) -> str:
    dynamic_suffix = f'\nTicker: {ticker}\nQuestion: {question}\n'
    return static_prefix + dynamic_suffix

key = prefix_cache_key(STATIC_PREFIX)
prompt = build_prompt(STATIC_PREFIX, 'NVDA', 'Give latest financial view and top risks.')
print('Prefix key:', key[:18], '...')
print(prompt[:300])

## 2) Semantic Caching with vector store reuse

This uses Chroma and Ollama embeddings.

We reuse the existing RAG vector DB directory and add a dedicated collection for semantic cache entries.

Flow:
1. embed incoming query
2. find semantically similar past query
3. if distance is good enough, return cached response and bypass LLM

In [ ]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)

semantic_cache = Chroma(
    collection_name='semantic_cache_langchain_finance',
    persist_directory=str(RAG_VECTOR_DB_DIR),
    embedding_function=embeddings,
)

def cache_response(query: str, answer: str, metadata: dict[str, Any] | None = None) -> None:
    metadata = metadata or {}
    metadata['cached_at'] = datetime.now(timezone.utc).isoformat()
    semantic_cache.add_documents([
        Document(page_content=query, metadata={**metadata, 'answer': answer})
    ])

def semantic_lookup(query: str, threshold: float = 0.12):
    hits = semantic_cache.similarity_search_with_score(query, k=1)
    if not hits:
        return None
    doc, dist = hits[0]
    if dist <= threshold:
        return {
            'query': doc.page_content,
            'answer': doc.metadata.get('answer', ''),
            'distance': float(dist),
            'metadata': doc.metadata,
        }
    return None

cache_response(
    query='What is the latest financial picture for NVDA?',
    answer='Cached answer example: NVDA remains tied to AI data center demand with volatility risk.',
    metadata={'ticker': 'NVDA'}
)

print(semantic_lookup('Give me a current financial overview of NVDA'))

## 3) Streamline and compress prompts (LLMLingua)

If LLMLingua is installed, compress large context before final generation.

Install option:
`pip install llmlingua`

In [ ]:
long_context = '\n'.join([
    'NVIDIA data center demand remains central to growth.',
    'Gross margins are sensitive to product mix and supply constraints.',
    'Macro and rate environment may impact valuation multiples.',
] * 30)

if PromptCompressor is None:
    print('LLMLingua not installed. Skipping compression demo.')
else:
    compressor = PromptCompressor()
    compressed = compressor.compress_prompt(long_context, rate=0.5)
    print('Original chars:', len(long_context))
    print('Compressed chars:', len(compressed))
    print(compressed[:500])

## 4) Concise instruction style

Avoid hedge language when you only need compact machine-useful outputs.

In [ ]:
CONCISE_SYSTEM = (
    'Reply tersely. No pleasantries. No hedging unless evidence is missing. '
    'Use bullet-like compact lines only.'
)

print(CONCISE_SYSTEM)

## 5) Compact output format + token constraints

Use compact serialization (YAML-like text) and strict output boundaries.

For Ollama in LangChain, `num_predict` is commonly used to bound generation length.

In [ ]:
yaml_prompt = '''
Return YAML only with keys:
ticker, trend, top_risks, confidence
'''

compact_llm = ChatOllama(
    model=MODEL_SMALL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
    num_predict=140,
)

print('Configured num_predict=140 for constrained generation.')

## 6) Task-based model routing

Use smaller models for easy tasks and larger models only when needed.

In [ ]:
def route_model(task: str) -> str:
    task_l = task.lower()
    if any(k in task_l for k in ['summary', 'classify', 'extract']):
        return MODEL_SMALL
    if any(k in task_l for k in ['reason', 'compare scenarios', 'deep analysis']):
        return MODEL_LARGE
    return MODEL_SMALL

for task in [
    'summary of latest headlines',
    'deep analysis of valuation scenarios',
    'extract key risks',
]:
    print(task, '->', route_model(task))

## 7) Batch processing

Batch non-dependent jobs to reduce orchestration overhead and improve throughput.

In [ ]:
def batch_fetch_snapshots(tickers: list[str]) -> list[dict[str, Any]]:
    snapshots = []
    for t in tickers:
        ticker_obj = yf.Ticker(t)
        hist = ticker_obj.history(period='1d', interval='1d')
        close = extract_latest_close(hist)
        snapshots.append({'ticker': t, 'latest_close': close})
    return snapshots

batch = batch_fetch_snapshots(['NVDA', 'AAPL', 'MSFT'])
print(json.dumps(batch, indent=2, ensure_ascii=True))

## 8) Integrated optimized query path

This function combines all optimization levers:
1. semantic cache lookup
2. broad evidence retrieval
3. rerank + compact context
4. concise + constrained generation
5. write-through semantic cache

In [ ]:
def optimized_answer(ticker: str, question: str) -> dict[str, Any]:
    cache_hit = semantic_lookup(question, threshold=0.12)
    if cache_hit:
        return {'path': 'semantic_cache', 'payload': cache_hit}

    snapshot = get_stock_snapshot.invoke({'ticker': ticker})
    snippets = build_evidence_snippets(snapshot)
    ranked = rerank_snippets(question, snippets)
    selected, used = select_minimal_context(ranked, token_budget=TOKEN_BUDGET, max_snippets=MAX_SNIPPETS)

    llm = ChatOllama(
        model=route_model('summary'),
        base_url=OLLAMA_BASE_URL,
        temperature=0,
        num_predict=140,
    )

    prompt = (
        f'{CONCISE_SYSTEM}\n\n'
        'Return YAML keys: ticker, trend, top_risks, confidence.\n'
        f'Ticker: {ticker}\nQuestion: {question}\n'
        'Evidence:\n' + '\n'.join(f'- {s}' for s in selected)
    )

    answer = llm.invoke(prompt).content
    cache_response(question, answer, metadata={'ticker': ticker, 'used_tokens': used})

    return {
        'path': 'llm',
        'used_tokens_estimate': used,
        'selected_snippets': selected,
        'answer': answer,
    }

result = optimized_answer('NVDA', 'Give latest financial picture and key risks.')
print(json.dumps(result if isinstance(result, dict) else {'answer': str(result)}, indent=2, ensure_ascii=True))